In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
data = pd.read_csv("../reviews_filtered.csv")
 # import reviews

print(data.head(10))

   Unnamed: 0  movie_id                       name  count_  user_id  rating
0           0     34222  slumber-party-massacre-ii      32   438333       7
1           1     34222  slumber-party-massacre-ii      32   437213       8
2           2     34222  slumber-party-massacre-ii      32   434710       7
3           3     34222  slumber-party-massacre-ii      32   423063       5
4           4     34222  slumber-party-massacre-ii      32   244269       9
5           5     34222  slumber-party-massacre-ii      32   188272       7
6           6     34222  slumber-party-massacre-ii      32    30514       6
7           7     34222  slumber-party-massacre-ii      32     8026      10
8           8    462065         run-sweetheart-run      32  1146451       5
9           9    462065         run-sweetheart-run      32  1067317       7


import re

love_death_robots_filter = re.compile("love-death-robots")
black_mirror_filter = re.compile("black-mirror")

to_be_removed = []

for idx, row in data.iterrows():
    if (re.match(love_death_robots_filter, row['name'])):
        to_be_removed.append(idx)
    elif (re.match(black_mirror_filter, row['name'])):
        to_be_removed.append(idx)

data = data.drop(to_be_removed)

data.to_csv("../reviews_filtered.csv")

In [3]:
unique_movies = sorted(list(set(zip(data['movie_id'], data['name']))))
unique_users = sorted(list(set(data['user_id'])))

num_movies = len(unique_movies)
num_users = len(unique_users)

print("%d Unique Movies" % num_movies)
print("%d Unique Users" % num_users)

5937 Unique Movies
2516 Unique Users


In [4]:
# these are a set of variables to make conversion between labels and attribute lookup easier
# users have two labels: their ids in the original dataset and their idxs in the list from the previous cell
# movies have 3: the same two as users, and also a name

movie_id_to_idx = {movie_id: idx for idx,
                   (movie_id, _) in enumerate(unique_movies)}
user_id_to_idx = {user_id: idx for idx, user_id in enumerate(unique_users)}

movie_name_to_idx = {movie_name: idx for idx,
                     (_, movie_name) in enumerate(unique_movies)}

movie_names = [movie_name for (_, movie_name) in unique_movies]

movie_id_to_name = {movie_id: movie_name for (
    movie_id, movie_name) in unique_movies}

# we also add a variable to easily look up how many reviews a movie has in our dataset
movie_freqs = {movie_id_to_idx[movie_id]: freq for (
    movie_id, freq) in zip(data['movie_id'], data['count_'])}


In [5]:
# loos up id and list idx for a movie given its name
# this uses fuzzy string matching so we can look up the attributes for a given movie
def look_up_movie(movie_name):
    match_string = re.compile(".*" + movie_name + ".*")
    for idx, (movie_id, movie_name) in enumerate(unique_movies):
        # movie_name = movie_id_to_name[movie_id]
        if (re.match(match_string, movie_name)):
            yield (movie_name, movie_id, idx, movie_freqs[idx])

In [6]:
# tests the above function
for (movie_name, movie_id, idx, freq) in look_up_movie("emma"):
    print(movie_name, movie_id, idx, freq)

emma-2020 485212 4103 454
the-social-dilemma 581924 4541 191


In [7]:
from surprise import Dataset, NormalPredictor, Reader


reader = Reader(rating_scale=(1, 10))
# the reader class in surprise just parses ratings on a given scale
# ours are from 1 to 10

dataset = Dataset.load_from_df(data[["user_id", "movie_id", "rating"]], reader)
# load data into surprise dataset class

In [8]:
from surprise import accuracy, Dataset, SVD
from surprise.model_selection import KFold

# tests the svd algorithm on the dataset as a baseline using 3 fold cross validation

# define a cross-validation iterator
kf = KFold(n_splits=3)

algo = SVD()

for trainset, testset in kf.split(dataset):

    # train and test svd algorithm.
    algo.fit(trainset)
    predictions = algo.test(testset)

    # Compute and print Root Mean Squared Error
    accuracy.rmse(predictions, verbose=True)

RMSE: 1.5439
RMSE: 1.5392
RMSE: 1.5410


In [14]:
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(dataset, random_state = 42)
algo = SVD()
algo.fit(trainset)
predictions = pd.DataFrame(algo.test(testset))


# runs svd algorithm on our dataset


In [ ]:
predictions["squared_error"] = (predictions["est"] - predictions["r_ui"]) *(predictions["est"] - predictions["r_ui"])  

mse_per_movie = predictions.groupby(["iid"]).agg({
     'squared_error': ['mean', 'count'], 
     'r_ui': ['mean', "std"], # actual ratings 
})

mse_per_movie['name'] = pd.Series(movie_id_to_name)

print(mse_per_movie.head(10))

# this gives us the mse average for each movie, so we can see if the algorithm is performing worse with reviews for some movies
# it does seem to be worse for some movies, specifically movies with lower average ratings and higher stdev of ratings
# but other than that the pattern is unclear
# this may be less relevant though, since for the purpose of recommendations
# we need to worry more about whether the high rating predictions are accurate
# false positives are worse than false negatives

    squared_error            r_ui                                       name
             mean count      mean       std                                 
iid                                                                         
247      1.894466    19  7.842105  1.708253                     take-shelter
277      3.169213    68  7.691176  2.001920                       drive-2011
278      3.737472    13  6.769231  1.877669              killing-them-softly
279      1.362284    43  7.232558  1.555898                   21-jump-street
281      5.455036    11  4.090909  2.256304                    47-ronin-2013
282      3.683999     6  6.166667  2.483277  extremely-loud-incredibly-close
285      4.882202    45  5.977778  2.168880            the-great-gatsby-2013
331      1.891001     6  5.166667  2.136976                total-recall-2012
603      1.716739    24  7.583333  1.691839                      the-muppets
994      1.169266     6  7.833333  1.169045                       jackass-35

In [ ]:
mse_per_user = predictions.groupby(["uid"]).agg({
     'squared_error': ['mean', 'count'], 
     'r_ui': ['mean', "std"], # actual ratings 
})

print(mse_per_user.head(10))

# this gives us the average prediction error for each user
# the prediction error seems to be about as correlated with specific movies as with users


    squared_error            r_ui          
             mean count      mean       std
uid                                        
1        1.979604     9  6.333333  3.427827
12       1.143405    16  6.312500  2.441823
24       1.032768    58  7.413793  2.077739
57       0.798891    46  5.608696  1.769849
60       2.361990     4  5.250000  2.629956
86       1.151385    95  6.578947  1.595065
88       1.392287     9  5.666667  1.414214
164      5.086923    54  6.777778  2.439198
167      0.604645    31  7.806452  1.621097
169      4.236061    43  5.302326  2.375819
